In [25]:
import treeswift as ts
import pandas as pd
import sys
import dendropy
from collections import defaultdict

In [26]:
G = 10 / 10**6

tns = dendropy.TaxonNamespace()
ttime = dendropy.Tree.get(
    path="../timetrees/63K_dated.tre",
    schema="newick",
    taxon_namespace=tns,
    preserve_underscores=True,
)
tcuo = dendropy.Tree.get(
    path="../63K.tre", schema="newick", taxon_namespace=tns, preserve_underscores=True
)
tsub = dendropy.Tree.get(
    path="../substrees/castlespro_stiller.tre",
    schema="newick",
    taxon_namespace=tns,
    preserve_underscores=True,
)

In [27]:
root = ttime.seed_node
l1, l2 = (nd.edge_length for nd in ttime.seed_node.child_nodes())

In [28]:
ttime.deroot()
tcuo.deroot()

for edge in tcuo.levelorder_edge_iter():
    if edge.length is None:
        edge.length = 0

length_diffs = dendropy.calculate.treecompare._get_length_diffs(ttime, tcuo)

i = 0
nd_ltime = [nd for nd in ttime.postorder_node_iter()]
nd_lcu = [nd for nd in tcuo.postorder_node_iter()]
nd_l = zip(nd_lcu, nd_ltime)
# root = None
lbl_to_popsize = {}
lbl_to_ngen = {}
lbl_to_cu = {}
lbl_to_time = {}
for nd_time in ttime.postorder_node_iter():
    if not nd_time.is_leaf():
        nd_time.label = f"N{i}"
        time, cu = length_diffs[i]
        if time == 0 and cu == 0:
            # root = nd_time
            for nd_child in nd_time.child_nodes():
                cu += lbl_to_cu[nd_child.label]
                time += lbl_to_time[nd_child.label]
                # print(time)
        g = time / G
        psize = g / cu
        # print(psize)
        lbl_to_time[nd_time.label] = time
        lbl_to_cu[nd_time.label] = cu
        lbl_to_popsize[nd_time.label] = psize
        for nd_child in nd_time.child_nodes():
            if nd_child.is_leaf():
                lbl_to_popsize[nd_child.taxon.label] = psize
    i += 1
ttime.scale_edges(1 / G)

In [29]:
ttime.deroot()
tsub.deroot()
length_diffs = dendropy.calculate.treecompare._get_length_diffs(ttime, tsub)
lbl_to_sub = {}
for i, nd in enumerate(ttime.postorder_node_iter()):
    time, sub = length_diffs[i]
    if nd.is_leaf():
        lbl_to_sub[nd.taxon.label] = sub
    else:
        lbl_to_sub[nd.label] = sub

In [30]:
ndr = [nd for nd in root.child_nodes()][0]
root = ttime.reroot_at_edge(ndr.edge, l2 / G, l1 / G)
root.label = "root"
lbl_to_popsize[root.label] = lbl_to_popsize[ndr.label]

In [31]:
for nd in ttime.postorder_node_iter():
    if nd.is_leaf():
        lbl_to_ngen[nd.taxon.label] = nd.edge_length
    else:
        lbl_to_ngen[nd.label] = nd.edge_length

In [32]:
subt = 0
lt = 0
for nd in root.child_nodes():
    subt += lbl_to_sub[nd.label]
    lt += nd.edge_length
for nd in root.child_nodes():
    lbl_to_sub[nd.label] = nd.edge_length / lt * subt
lbl_to_sub[root.label] = 0

In [33]:
with open("../main-num_generations.tre", "w") as dest:
    ttime.write(file=dest, schema="newick")

In [34]:
ann_to_lbls = defaultdict(list)
with open("../annotation.txt", "r") as f:
    for l in f.readlines():
        lbl, ann = l.strip().split()
        ann_to_lbls[ann].append(lbl)

In [35]:
ttime = ts.read_tree_newick("../main-num_generations.tre")
lbl_to_nd = ttime.label_to_node(selection="all")
lbl_to_ann = {}
for ann, lbls in ann_to_lbls.items():
    if len(lbls) > 1:
        nd = ttime.mrca(lbls)
        lbl_to_ann[nd.get_label()] = ann
        nd.set_label(ann)
ttime.write_tree_newick("../main-num_generations.tre")

In [215]:
with open("../num-generations.tsv", "w") as f:
    f.write(f"LABEL\tNGEN")
    for lbl, ngen in lbl_to_ngen.items():
        lbl = lbl_to_ann.get(lbl, lbl)
        f.write(f"\n{lbl}\t{ngen}")

In [216]:
with open("../population-sizes.tsv", "w") as f:
    f.write(f"LABEL\tSIZE")
    for lbl, size in lbl_to_popsize.items():
        lbl = lbl_to_ann.get(lbl, lbl)
        f.write(f"\n{lbl}\t{size}")

In [217]:
with open("../substitution-units.tsv", "w") as f:
    f.write(f"LABEL\tSRATE")
    for lbl, sub in lbl_to_sub.items():
        lbl = lbl_to_ann.get(lbl, lbl)
        f.write(f"\n{lbl}\t{sub}")

In [37]:
node_to_depth_ngen = {}
node_to_depth_edge = {}
for node, depth in ttime.distances_from_root(
    leaves=True, internal=True, unlabeled=False, weighted=True
):
    node_to_depth_ngen[node.get_label()] = depth
for node, depth in ttime.distances_from_root(
    leaves=True, internal=True, unlabeled=False, weighted=False
):
    node_to_depth_edge[node.get_label()] = depth
with open("../population-information.tsv", "w") as f:
    f.write(
        f"LABEL\tSRATE\tSIZE\tNGEN\tCLADE_SIZE\tHEIGHT_NGEN\tHEIGHT_EDGE\tDEPTH_NGEN\tDEPTH_EDGE\tTOTAL_NGEN"
    )
    for label, ngen in lbl_to_sub.items():
        ann = lbl_to_ann.get(label, label)
        node = lbl_to_nd[label]
        size = lbl_to_popsize[label]
        srate = lbl_to_sub[label]
        ngen = lbl_to_ngen[label]
        subtree = ttime.extract_subtree(node)
        clade_size = subtree.num_nodes(internal=False)
        height_ngen = subtree.height()
        height_edge = subtree.height(weighted=False)
        total_ngen = subtree.edge_length_sum(terminal=True, internal=True)
        depth_ngen = node_to_depth_ngen[ann]
        depth_edge = node_to_depth_edge[ann]
        f.write(
            f"{ann}\t{srate}\t{size}\t{ngen}\t{clade_size}\t{height_ngen}\t{height_edge}\t{depth_ngen}\t{depth_edge}\t{total_ngen}\n"
        )